# Can we predict 12-month loan default?
Follows `checklist.md`. Target: `defaulted`. Data: 8000 applications, one row per `application_id`.

**Data facts that drive the design**
- All feature columns are recorded at application (per `columns.csv`), so all are available at prediction time. `application_id` is an identifier and is dropped.
- There is no date column and ids are not ordered/correlated with the outcome, so no time order is available; and every `application_id` is unique (no repeated entities). A random stratified split is therefore the appropriate structure. If rows are in fact time-ordered in production, this should be revisited.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, brier_score_loss

df = pd.read_csv('data.csv')
assert df.application_id.is_unique and df.isna().sum().sum()==0
print(df.shape, df.defaulted.mean())

## 1. Set aside the test set (before any modelling)

In [ ]:
y = df['defaulted']
X = df.drop(columns=['defaulted','application_id'])   # identifier dropped
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(len(X_tr), len(X_te), y_tr.mean(), y_te.mean())
# X_te / y_te are not touched again until step 5.

## 2. Candidate models (named before scoring any)
1. `baseline` - predicts the training prior (trivial).
2. `logreg` - standardised numerics + one-hot region, L2 logistic regression, C in {0.1, 1, 10}.
3. `hgb` - gradient boosting, learning_rate in {0.03, 0.1}, max_depth in {2, 3}.

Selection metric: cross-validated average precision (PR-AUC; ~10% positive rate). Decision threshold: chosen on out-of-fold predictions of the winner to maximise F1. All features are used (no feature selection).

In [ ]:
num = ['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
def make(kind, **p):
    if kind=='baseline': return DummyClassifier(strategy='prior')
    if kind=='logreg':
        pre = ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),['region'])])
        return Pipeline([('pre',pre),('m',LogisticRegression(C=p['C'],max_iter=1000))])
    pre = ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore'),['region'])])
    return Pipeline([('pre',pre),('m',HistGradientBoostingClassifier(learning_rate=p['lr'],max_depth=p['d'],max_iter=200,random_state=0))])
cands = {'baseline':('baseline',{})}
for C_ in [0.1,1,10]: cands[f'logreg C={C_}']=('logreg',{'C':C_})
for lr in [0.03,0.1]:
    for d in [2,3]: cands[f'hgb lr={lr} d={d}']=('hgb',{'lr':lr,'d':d})

## 3. Cross-validation on training rows only (preprocessing is inside each pipeline, refit per fold)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=1)
rows, oof = [], {}
for name,(k,p) in cands.items():
    pr = cross_val_predict(make(k,**p), X_tr, y_tr, cv=cv, method='predict_proba')[:,1]
    oof[name]=pr
    fold_ap=[average_precision_score(y_tr.iloc[v],pr[v]) for _,v in cv.split(X_tr,y_tr)]
    rows.append((name, np.mean(fold_ap), np.std(fold_ap), roc_auc_score(y_tr,pr)))
cvres = pd.DataFrame(rows,columns=['model','cv_AP','AP_sd','oof_AUC']).sort_values('cv_AP',ascending=False)
cvres

## 4. Choose model and threshold on CV only

In [ ]:
best = cvres.iloc[0].model
ths = np.linspace(0.02,0.6,59)
f1s = [f1_score(y_tr, oof[best]>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr,3), '| CV F1:', round(max(f1s),3))

**Decision (recorded before test scoring):** the model and threshold printed above are final. The test set is scored once below, with no further changes.

## 5. Refit on all training rows, score the test set once

In [ ]:
k,p = cands[best]
final = make(k,**p).fit(X_tr,y_tr)
pt = final.predict_proba(X_te)[:,1]; pred = pt>=thr
base = make('baseline').fit(X_tr,y_tr).predict_proba(X_te)[:,1]
res = pd.DataFrame({'chosen':[roc_auc_score(y_te,pt),average_precision_score(y_te,pt),precision_score(y_te,pred),recall_score(y_te,pred),f1_score(y_te,pred),brier_score_loss(y_te,pt)],
                    'baseline':[roc_auc_score(y_te,base),average_precision_score(y_te,base),0,0,0,brier_score_loss(y_te,base)]},
                   index=['ROC AUC','PR AUC','precision@thr','recall@thr','F1@thr','Brier'])
res.round(3)

In [ ]:
# uncertainty: bootstrap CI on test AUC / AP (model fixed)
rng=np.random.default_rng(0); yt=y_te.values; a=[];b=[]
for _ in range(1000):
    i=rng.integers(0,len(yt),len(yt)); a.append(roc_auc_score(yt[i],pt[i])); b.append(average_precision_score(yt[i],pt[i]))
print('AUC 95% CI', np.percentile(a,[2.5,97.5]).round(3), '| AP 95% CI', np.percentile(b,[2.5,97.5]).round(3), '| base rate', yt.mean().round(3))

## Conclusion
See the test-set table above (the headline), compared against the prior-only baseline and the bootstrap interval. Caveats: no timestamps, so generalisation to later cohorts is untested; the test set has ~200 positives, so metrics are noisy.